# Phase 2 — Static GCN Baseline
**Project:** Temporal GNN + XAI for Anti-Money Laundering  
**Model:** Static GCN (2-layer GCNConv, 165 → 64 → 32 → 2)  
**Purpose:** Establish drift-vulnerable baseline. Compare against EvolveGCN-H in Phase 3.

### Temporal Split
| Split | Snapshots | Indices (0-based) |
|-------|-----------|-------------------|
| Train | T1 – T34  | 0 – 33            |
| Val   | T35 – T36 | 34 – 35           |
| Test  | T37 – T49 | 36 – 48           |

### Cells in this notebook
1. Imports & global config  
2. Device setup  
3. Load snapshots  
4. Verify splits  
5. Model definition  
6. Loss function & class weights  
7. Training & evaluation functions  
8. Main training loop with early stopping  
9. Load best model & full test evaluation  
10. Per-snapshot F1 & AUROC over test period  
11. Drift probe (train T1–T16, test T33–T49)  
12. Save all results

---
## Cell 1 — Imports & Global Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjust BASE_DIR if running on Google Colab:
# BASE_DIR = '/content/drive/MyDrive/Capstone'
BASE_DIR      = '/content/drive/MyDrive/Capstone/AML Code'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
FIGURES_DIR   = os.path.join(BASE_DIR, 'figures')

for d in [MODELS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Temporal split (0-based indices into snapshots list) ──────────────────────
# snapshots[0]  = T1    (time_step 1)   ← first training snapshot
# snapshots[33] = T34   (time_step 34)  ← last training snapshot
# snapshots[34] = T35   (time_step 35)  ← first val snapshot
# snapshots[35] = T36   (time_step 36)  ← last val snapshot
# snapshots[36] = T37   (time_step 37)  ← first test snapshot
# snapshots[48] = T49   (time_step 49)  ← last test snapshot
TRAIN_IDX = list(range(0,  34))   # T1  – T34  (34 snapshots)
VAL_IDX   = list(range(34, 36))   # T35 – T36  ( 2 snapshots)
TEST_IDX  = list(range(36, 49))   # T37 – T49  (13 snapshots)

# ── Model hyper-parameters ────────────────────────────────────────────────────
IN_CHANNELS  = 165
HIDDEN1      = 64
HIDDEN2      = 32
DROPOUT      = 0.5
LR           = 0.001
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 200
PATIENCE     = 20        # early stopping on val AUROC
CLASS_W_ILLICIT = 9.0   # ~9:1 licit:illicit imbalance

# ── Drift probe split ─────────────────────────────────────────────────────────
DRIFT_TRAIN_IDX = list(range(0,  16))   # T1  – T16
DRIFT_TEST_IDX  = list(range(32, 49))   # T33 – T49

# ── Dark market shutdown ──────────────────────────────────────────────────────
SHUTDOWN_STEP = 43

print('Imports complete.')
print(f'Train : T1  - T34  ({len(TRAIN_IDX)} snapshots,  indices {TRAIN_IDX[0]}–{TRAIN_IDX[-1]})')
print(f'Val   : T35 - T36  ({len(VAL_IDX)} snapshots,  indices {VAL_IDX[0]}–{VAL_IDX[-1]})')
print(f'Test  : T37 - T49  ({len(TEST_IDX)} snapshots, indices {TEST_IDX[0]}–{TEST_IDX[-1]})')

---
## Cell 2 — Device Setup

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# Seed CUDA if available
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

---
## Cell 3 — Load Snapshots

In [ ]:
snapshots_path = os.path.join(PROCESSED_DIR, 'snapshots.pt')
snapshots = torch.load(snapshots_path, weights_only=False)

print(f'Snapshots loaded  : {len(snapshots)}')
print(f'Feature dimension : {snapshots[0].x.shape[1]}  (expected: 165)')
print(f'Unique labels     : {torch.unique(snapshots[0].y).tolist()}  (expected: [0, 1])')
print()

# Quick sanity check on every snapshot
for s in snapshots:
    assert s.x.shape[1] == IN_CHANNELS, f'T{s.time_step.item()} has wrong feature dim: {s.x.shape[1]}'
    assert not torch.isnan(s.x).any(), f'T{s.time_step.item()} has NaN features'

print('All 49 snapshots passed sanity checks.')

---
## Cell 4 — Verify Splits

In [ ]:
print('SPLIT VERIFICATION')
print('=' * 50)

for split_name, idx_list in [('TRAIN', TRAIN_IDX), ('VAL', VAL_IDX), ('TEST', TEST_IDX)]:
    snaps = [snapshots[i] for i in idx_list]
    t_steps = [s.time_step.item() for s in snaps]
    total_nodes   = sum(s.num_nodes for s in snaps)
    total_illicit = sum(s.y.sum().item() for s in snaps)
    print(f'{split_name:6} | T{t_steps[0]}–T{t_steps[-1]}'
          f' | {len(idx_list)} snapshots'
          f' | {total_nodes:,} nodes'
          f' | {int(total_illicit):,} illicit ({total_illicit/total_nodes*100:.1f}%)')

print()
# Confirm no index overlap between splits
assert not set(TRAIN_IDX) & set(VAL_IDX),  'TRAIN and VAL overlap!'
assert not set(TRAIN_IDX) & set(TEST_IDX), 'TRAIN and TEST overlap!'
assert not set(VAL_IDX)   & set(TEST_IDX), 'VAL and TEST overlap!'
print('No split overlap confirmed. Ready to train.')

---
## Cell 5 — Model Definition

In [ ]:
class StaticGCN(nn.Module):
    """
    2-layer Graph Convolutional Network.
    Architecture: 165 → 64 → 32 → 2
    No temporal awareness — weights are fixed after training.
    Used as a drift-vulnerable baseline against EvolveGCN-H.
    """
    def __init__(self,
                 in_channels  = IN_CHANNELS,
                 hidden1      = HIDDEN1,
                 hidden2      = HIDDEN2,
                 dropout      = DROPOUT):
        super().__init__()
        self.conv1      = GCNConv(in_channels, hidden1)
        self.conv2      = GCNConv(hidden1, hidden2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2, 2)

    def forward(self, x, edge_index):
        # Layer 1
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        # Layer 2
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        embeddings = x                      # 32-dim node embeddings
        logits = self.classifier(x)         # 2-dim output
        return logits, embeddings

# Quick parameter count
model = StaticGCN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'StaticGCN architecture:')
print(f'  GCNConv : {IN_CHANNELS} → {HIDDEN1}')
print(f'  GCNConv : {HIDDEN1} → {HIDDEN2}')
print(f'  Linear  : {HIDDEN2} → 2')
print(f'  Dropout : {DROPOUT}')
print(f'  Total trainable parameters: {n_params:,}')

---
## Cell 6 — Loss Function & Optimizer

In [ ]:
# Weighted CrossEntropy to handle ~9:1 class imbalance
# Weight 1.0 for licit (majority), 9.0 for illicit (minority)
class_weights = torch.tensor([1.0, CLASS_W_ILLICIT]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY
)

print(f'Loss     : CrossEntropyLoss  weights=[1.0 (licit), {CLASS_W_ILLICIT} (illicit)]')
print(f'Optimizer: AdamW  lr={LR}  weight_decay={WEIGHT_DECAY}')

---
## Cell 7 — Training & Evaluation Functions

In [ ]:
def train_epoch(mdl, optim, idx_list):
    """
    One training epoch: iterate over all snapshots in idx_list.
    Each snapshot is processed as a full graph (no mini-batching).
    """
    mdl.train()
    total_loss = 0.0

    for i in idx_list:
        data = snapshots[i].to(device)
        optim.zero_grad()
        out, _ = mdl(data.x, data.edge_index)
        loss = criterion(out, data.y)
        loss.backward()
        optim.step()
        total_loss += loss.item()

    return total_loss / len(idx_list)


def evaluate(mdl, idx_list):
    """
    Evaluate model on a list of snapshot indices.
    Returns: F1 (illicit), AUROC, Precision, Recall
    """
    mdl.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for i in idx_list:
            data = snapshots[i].to(device)
            out, _ = mdl(data.x, data.edge_index)
            probs = torch.softmax(out, dim=1)[:, 1]   # probability of illicit
            preds = out.argmax(dim=1)

            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    # Guard against snapshots with only one class (AUROC undefined)
    if len(set(y_true)) < 2:
        auroc = float('nan')
    else:
        auroc = roc_auc_score(y_true, y_prob)

    return {
        'F1'       : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'AUROC'    : auroc,
        'Precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Recall'   : recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }


def evaluate_per_snapshot(mdl, idx_list):
    """
    Returns per-snapshot F1 and AUROC — used for drift visualisation.
    """
    results = []
    for i in idx_list:
        metrics = evaluate(mdl, [i])
        t = snapshots[i].time_step.item()
        results.append({'time_step': t, **metrics})
    return results


print('Training and evaluation functions defined.')

---
## Cell 8 — Main Training Loop with Early Stopping

In [ ]:
# Re-initialise model and optimizer fresh
torch.manual_seed(SEED)
model     = StaticGCN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_auroc = 0.0
patience_counter = 0
train_loss_history = []
val_auroc_history  = []
val_f1_history     = []

print(f'Training StaticGCN for up to {MAX_EPOCHS} epochs (early stop patience={PATIENCE})')
print(f'Train: T1–T34 ({len(TRAIN_IDX)} snapshots)   Val: T35–T36 ({len(VAL_IDX)} snapshots)')
print(f'Early stopping criterion: Val AUROC')
print('-' * 65)
print(f'{"Epoch":>6}  {"Train Loss":>11}  {"Val F1":>8}  {"Val AUROC":>10}  {"Status"}')
print('-' * 65)

for epoch in range(MAX_EPOCHS):
    loss    = train_epoch(model, optimizer, TRAIN_IDX)
    val_met = evaluate(model, VAL_IDX)

    val_f1    = val_met['F1']
    val_auroc = val_met['AUROC']

    train_loss_history.append(loss)
    val_auroc_history.append(val_auroc)
    val_f1_history.append(val_f1)

    # Early stopping on AUROC
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        patience_counter = 0
        torch.save(model.state_dict(),
                   os.path.join(MODELS_DIR, 'static_gcn_best.pt'))
        status = '✓ saved'
    else:
        patience_counter += 1
        status = f'patience {patience_counter}/{PATIENCE}'

    if epoch % 10 == 0 or patience_counter == PATIENCE:
        print(f'{epoch:>6}  {loss:>11.4f}  {val_f1:>8.4f}  {val_auroc:>10.4f}  {status}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest Val AUROC : {best_val_auroc:.4f}')
print(f'Model saved to : models/static_gcn_best.pt')

---
## Cell 9 — Training Curve Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_ran = range(len(train_loss_history))

axes[0].plot(epochs_ran, train_loss_history, color='#4A90D9', linewidth=2)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, val_auroc_history, color='#0D9488', linewidth=2, label='Val AUROC')
axes[1].plot(epochs_ran, val_f1_history,    color='#F59E0B', linewidth=2, label='Val F1 (illicit)')
axes[1].axhline(best_val_auroc, color='#0D9488', linestyle=':', alpha=0.5,
                label=f'Best AUROC = {best_val_auroc:.4f}')
axes[1].set_title('Validation Metrics', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Static GCN — Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'static_gcn_training_curves.png'), bbox_inches='tight')
plt.show()
print('Saved → figures/static_gcn_training_curves.png')

---
## Cell 10 — Load Best Model & Full Test Evaluation

In [ ]:
# Load the best checkpoint saved during training
model.load_state_dict(
    torch.load(os.path.join(MODELS_DIR, 'static_gcn_best.pt'),
               map_location=device, weights_only=True)
)
model.eval()
print('Best model loaded.')

# ── Full test evaluation ──────────────────────────────────────────────────────
test_metrics = evaluate(model, TEST_IDX)

print()
print('STATIC GCN — FULL TEST RESULTS  (T37 – T49)')
print('=' * 45)
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v:.4f}')
print('=' * 45)
print()
print('NOTE: These aggregate metrics hide drift.')
print('Per-snapshot breakdown in the next cell reveals the T43 collapse.')

---
## Cell 11 — Per-Snapshot Performance Over Test Period

In [ ]:
per_snap = evaluate_per_snapshot(model, TEST_IDX)

print('PER-SNAPSHOT RESULTS — T37 to T49')
print(f'{"T":>4}  {"F1":>8}  {"AUROC":>8}  {"Precision":>10}  {"Recall":>8}  Note')
print('-' * 65)
for r in per_snap:
    note = '← SHUTDOWN' if r['time_step'] == SHUTDOWN_STEP else ''
    auroc_str = f"{r['AUROC']:.4f}" if not (isinstance(r['AUROC'], float) and r['AUROC'] != r['AUROC']) else 'nan  '
    print(f"T{r['time_step']:>2}  {r['F1']:>8.4f}  {auroc_str:>8}  "
          f"{r['Precision']:>10.4f}  {r['Recall']:>8.4f}  {note}")

# Compute pre- vs post-shutdown averages
pre  = [r['F1'] for r in per_snap if r['time_step'] < SHUTDOWN_STEP]
post = [r['F1'] for r in per_snap if r['time_step'] > SHUTDOWN_STEP]

print()
print(f'Mean F1 pre-T43  (T37-T42) : {np.mean(pre):.4f}')
print(f'Mean F1 post-T43 (T44-T49) : {np.mean(post):.4f}')
print(f'Drop                       : {np.mean(pre) - np.mean(post):.4f}')

---
## Cell 12 — Drift Visualisation Plot

In [ ]:
t_steps = [r['time_step'] for r in per_snap]
f1s     = [r['F1']        for r in per_snap]
aurocs  = [r['AUROC']     for r in per_snap]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, vals, ylabel, title in [
    (axes[0], f1s,    'F1 Score (Illicit Class)', 'F1 Score Over Test Period'),
    (axes[1], aurocs, 'AUROC',                    'AUROC Over Test Period'),
]:
    ax.plot(t_steps, vals, marker='o', linewidth=2,
            color='#4A90D9', markersize=7, label='Static GCN')
    ax.axvline(x=SHUTDOWN_STEP, color='#E05252', linestyle='--',
               linewidth=2, label=f'Dark Market Shutdown (T{SHUTDOWN_STEP})')
    ax.axvspan(37, SHUTDOWN_STEP, alpha=0.05, color='blue',  label='Pre-shutdown')
    ax.axvspan(SHUTDOWN_STEP, 49, alpha=0.05, color='red',   label='Post-shutdown')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Time Snapshot')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xticks(t_steps)

plt.suptitle('Static GCN — Concept Drift at T43 (Dark Market Shutdown)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'static_gcn_drift.png'), bbox_inches='tight')
plt.show()
print('Saved → figures/static_gcn_drift.png')

---
## Cell 13 — Drift Probe
Train on T1–T16 only, then test on T33–T49.  
This exaggerates the drift by using an even shorter training window,  
making the concept drift signal clearer for the paper.

In [ ]:
print('DRIFT PROBE — Train: T1–T16  |  Test: T33–T49')
print('=' * 55)

# Fresh model for drift probe
torch.manual_seed(SEED)
drift_model     = StaticGCN().to(device)
drift_optimizer = torch.optim.AdamW(
    drift_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)

# Train on T1–T16 (no validation / early stopping for probe)
drift_model.train()
for epoch in range(MAX_EPOCHS):
    loss = train_epoch(drift_model, drift_optimizer, DRIFT_TRAIN_IDX)
    if epoch % 50 == 0:
        print(f'  Epoch {epoch:>3}  Loss: {loss:.4f}')

# Per-snapshot evaluation on T33–T49
probe_results = evaluate_per_snapshot(drift_model, DRIFT_TEST_IDX)

print()
print(f'{"T":>4}  {"F1":>8}  {"AUROC":>8}  Note')
print('-' * 40)
for r in probe_results:
    note = '← SHUTDOWN' if r['time_step'] == SHUTDOWN_STEP else ''
    auroc_str = f"{r['AUROC']:.4f}" if not (isinstance(r['AUROC'], float) and r['AUROC'] != r['AUROC']) else 'nan  '
    print(f"T{r['time_step']:>2}  {r['F1']:>8.4f}  {auroc_str:>8}  {note}")

pre_probe  = [r['F1'] for r in probe_results if r['time_step'] < SHUTDOWN_STEP]
post_probe = [r['F1'] for r in probe_results if r['time_step'] > SHUTDOWN_STEP]
print()
print(f'Mean F1 pre-T43  : {np.mean(pre_probe):.4f}')
print(f'Mean F1 post-T43 : {np.mean(post_probe):.4f}')
print(f'Drop             : {np.mean(pre_probe) - np.mean(post_probe):.4f}')

---
## Cell 14 — Drift Probe Plot

In [ ]:
probe_t  = [r['time_step'] for r in probe_results]
probe_f1 = [r['F1']        for r in probe_results]

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(probe_t, probe_f1, marker='o', linewidth=2,
        color='#E05252', markersize=7, label='Static GCN (trained T1–T16)')
ax.axvline(x=SHUTDOWN_STEP, color='darkred', linestyle='--',
           linewidth=2.5, label=f'Dark Market Shutdown (T{SHUTDOWN_STEP})')
ax.axvspan(33, SHUTDOWN_STEP, alpha=0.05, color='blue')
ax.axvspan(SHUTDOWN_STEP, 49, alpha=0.05, color='red')
ax.text(35, ax.get_ylim()[0]+0.02, 'Pre-shutdown', fontsize=9, color='#2255AA')
ax.text(44, ax.get_ylim()[0]+0.02, 'Post-shutdown', fontsize=9, color='#AA2222')

ax.set_title('Drift Probe — Static GCN Trained on T1–T16, Tested on T33–T49',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Time Snapshot')
ax.set_ylabel('F1 Score (Illicit Class)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(probe_t)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'static_gcn_drift_probe.png'), bbox_inches='tight')
plt.show()
print('Saved → figures/static_gcn_drift_probe.png')

---
## Cell 15 — Save Outputs for Phase 4 (SHAP)

In [ ]:
# Save predictions and embeddings for all test snapshots
# These will be loaded by the SHAP notebook in Phase 4
model.eval()
all_outputs = {}

with torch.no_grad():
    for i in TEST_IDX:
        data = snapshots[i].to(device)
        out, emb = model(data.x, data.edge_index)
        probs = torch.softmax(out, dim=1)[:, 1]

        all_outputs[i] = {
            'time_step'  : snapshots[i].time_step.item(),
            'preds'      : out.argmax(dim=1).cpu(),
            'probs'      : probs.cpu(),
            'embeddings' : emb.cpu(),
            'y_true'     : data.y.cpu(),
        }

output_path = os.path.join(PROCESSED_DIR, 'static_gcn_outputs.pt')
torch.save(all_outputs, output_path)
print(f'Outputs saved → {output_path}')

# Save summary metrics to JSON for paper table
summary = {
    'model'           : 'Static GCN',
    'architecture'    : f'{IN_CHANNELS} → {HIDDEN1} → {HIDDEN2} → 2',
    'train_snapshots' : 'T1-T34',
    'val_snapshots'   : 'T35-T36',
    'test_snapshots'  : 'T37-T49',
    'best_val_auroc'  : round(best_val_auroc, 4),
    'test_metrics'    : {k: round(v, 4) for k, v in test_metrics.items()},
    'per_snapshot'    : per_snap,
    'drift_probe'     : {
        'train' : 'T1-T16',
        'test'  : 'T33-T49',
        'mean_f1_pre_t43'  : round(float(np.mean(pre_probe)), 4),
        'mean_f1_post_t43' : round(float(np.mean(post_probe)), 4),
        'f1_drop'          : round(float(np.mean(pre_probe) - np.mean(post_probe)), 4),
    }
}

summary_path = os.path.join(PROCESSED_DIR, 'static_gcn_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary saved  → {summary_path}')

print()
print('PHASE 2 COMPLETE')
print('Next step: Phase 3 — EvolveGCN-H training')
print('Compare drift probe F1 drop against Static GCN to quantify temporal benefit.')